# Instagram top accounts: quick look

Loads the daily follower counts and plots the 10 biggest accounts and the biggest 7-day moves.

Data: [FastSocial Top Charts](https://fastsocial.co/most-followed-instagram-accounts), CC BY 4.0.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

path = glob.glob('/kaggle/input/**/daily.csv', recursive=True)[0]
df = pd.read_csv(path, parse_dates=['date'])
df.tail()

## The 10 biggest accounts on the latest day

In [ ]:
last = df['date'].max()
top = df[df['date'] == last].nlargest(10, 'followers').sort_values('followers')

ax = top.plot.barh(x='name', y='followers', legend=False, figsize=(8, 5))
ax.set_xlabel('Followers (millions)')
ax.xaxis.set_major_formatter(lambda v, _: f'{v / 1e6:.0f}')
ax.set_ylabel('')
ax.set_title(f'Most followed Instagram accounts, {last:%d %b %Y}')
plt.tight_layout()
plt.show()

## Biggest 7-day moves

Only accounts read on both days are compared.

In [ ]:
start = last - pd.Timedelta(days=7)
if start not in set(df['date']):
    start = df['date'].min()
a = df[df['date'] == start].set_index('username')['followers']
b = df[df['date'] == last].set_index('username')
change = (b['followers'] - a).dropna().astype(int)
moves = pd.concat([change.nlargest(5), change.nsmallest(5)]).sort_values()
labels = b.loc[moves.index, 'name']

ax = moves.plot.barh(figsize=(8, 5), color=['#c0392b' if v < 0 else '#2e7d32' for v in moves])
ax.set_yticklabels(labels)
ax.set_xlabel('Follower change')
ax.set_title(f'Biggest gains and losses, {start:%d %b} to {last:%d %b %Y}')
plt.tight_layout()
plt.show()